# 02 — Silver Cleaning

Cleans bronze tables and resolves cross-system identity into a set of conformed Silver entities.

The key output is `silver_student` — a master record that spans all three source systems.

## Identity Resolution Strategy

| Link | Key | Notes |
|---|---|---|
| HubSpot Contact → D365 Student | `d365.contact_id = hs.contact_id` | Direct FK. ~10% of D365 students are walk-ins with no HubSpot record (NULL `contact_id`). |
| D365 Student → Paradigm Student | `LOWER(TRIM(d365.email)) = LOWER(TRIM(paradigm.email))` | Email-based. ~17% of D365 students have no Paradigm record. |
| D365 Course → Paradigm Unit | `d365.course_code = paradigm.course_code` | Direct string match. |
| D365 Intake → Paradigm Result | `d365.intake_code = paradigm.intake_code` | Direct string match. |

All joins use LEFT JOIN from D365 as the spine so walk-ins and students without results are preserved.

## silver_student — Master Entity

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE silver_student AS
    SELECT
        d.student_number,
        d.contact_id,
        h.marketing_source_id,
        p.student_code,
        d.first_name,
        d.last_name,
        LOWER(TRIM(d.email))   AS email,
        d.phone,
        d.date_of_birth,
        d.address_suburb,
        d.address_state,
        d.lifecycle_status,
        d.enrolment_date,
        h.lifecycle_stage      AS hs_lifecycle_stage,
        h.created_date         AS hs_created_date
    FROM bronze_dynamics365_students d
    LEFT JOIN bronze_hubspot_contacts h
        ON d.contact_id = h.contact_id
    LEFT JOIN bronze_paradigm_students p
        ON LOWER(TRIM(d.email)) = LOWER(TRIM(p.email))
""")
print(f"silver_student: {spark.table('silver_student').count()} rows")


## Supporting Silver Tables

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE silver_marketing_source AS
    SELECT
        marketing_source_id,
        source_name,
        source_category
    FROM bronze_hubspot_marketing_sources
""")

spark.sql("""
    CREATE OR REPLACE TABLE silver_activity AS
    SELECT
        activity_id,
        contact_id,
        marketing_source_id,
        activity_type,
        CAST(activity_date AS DATE) AS activity_date,
        activity_detail
    FROM bronze_hubspot_activities
""")

spark.sql("""
    CREATE OR REPLACE TABLE silver_course AS
    SELECT
        course_id,
        course_code,
        course_name,
        course_level,
        CAST(duration_terms AS INTEGER) AS duration_terms
    FROM bronze_dynamics365_courses
""")

spark.sql("""
    CREATE OR REPLACE TABLE silver_intake AS
    SELECT
        intake_id,
        intake_code,
        CAST(start_date AS DATE)      AS start_date,
        CAST(end_date AS DATE)        AS end_date,
        CAST(academic_year AS INTEGER) AS academic_year,
        term
    FROM bronze_dynamics365_intakes
""")

for tbl in ["silver_marketing_source", "silver_activity", "silver_course", "silver_intake"]:
    print(f"{tbl}: {spark.table(tbl).count()} rows")


In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE silver_enrolment AS
    SELECT
        enrolment_id,
        student_number,
        course_id,
        intake_id,
        CAST(enrolment_date AS DATE)                      AS enrolment_date,
        status,
        CAST(NULLIF(TRIM(final_grade), '') AS INTEGER)    AS final_grade
    FROM bronze_dynamics365_enrolments
""")
print(f"silver_enrolment: {spark.table('silver_enrolment').count()} rows")


In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE silver_unit AS
    SELECT
        unit_id,
        unit_code,
        unit_name,
        course_code
    FROM bronze_paradigm_units
""")

# Resolve Paradigm student_code to D365 student_number via silver_student
spark.sql("""
    CREATE OR REPLACE TABLE silver_result AS
    SELECT
        r.result_id,
        r.student_code,
        ss.student_number,
        r.unit_id,
        u.course_code,
        r.intake_code,
        CAST(r.mark AS INTEGER)         AS mark,
        r.grade,
        CAST(r.result_date AS DATE)     AS result_date
    FROM bronze_paradigm_results r
    JOIN silver_student  ss ON r.student_code = ss.student_code
    JOIN silver_unit      u ON r.unit_id       = u.unit_id
""")

for tbl in ["silver_unit", "silver_result"]:
    print(f"{tbl}: {spark.table(tbl).count()} rows")


## Verification

In [ ]:
silver_tables = [
    "silver_student",
    "silver_marketing_source",
    "silver_activity",
    "silver_course",
    "silver_intake",
    "silver_enrolment",
    "silver_unit",
    "silver_result",
]

print(f"{'Table':<30} {'Rows':>6}")
print("-" * 38)
for tbl in silver_tables:
    print(f"{tbl:<30} {spark.table(tbl).count():>6}")

# Identity resolution coverage
total = spark.table("silver_student").count()
linked_hs = spark.sql("SELECT COUNT(*) FROM silver_student WHERE contact_id IS NOT NULL").collect()[0][0]
linked_paradigm = spark.sql("SELECT COUNT(*) FROM silver_student WHERE student_code IS NOT NULL").collect()[0][0]
print(f"\nIdentity resolution:")
print(f"  HubSpot link:  {linked_hs}/{total} ({linked_hs/total*100:.1f}%)")
print(f"  Paradigm link: {linked_paradigm}/{total} ({linked_paradigm/total*100:.1f}%)")
